# The FACE cohorts together: the common instruments

The bipolar (`bp`), schizophrenia (`sz`), resistant depression (`dr`) and autism spectrum (`asp`) cohorts are loaded by the same package, `facedata`. This notebook shows how to put them in one table for transdiagnostic work. It is written for people who do not code much.

You will learn how to:

1. read the **catalogue** of the instruments that two cohorts or more share, and what its verdict means;
2. load these instruments from the four extractions;
3. put the **screening visit and the inclusion visit in one row** per patient;
4. **stack** the four cohorts, and join them the right way;
5. **count before you compare**, look at the years, and read the flags;
6. make a few comparisons, add the follow-up visits, and save your table.

The document `reports/executive/FACE-common-instruments.html` describes every shared instrument, cohort by cohort. Keep it open next to this notebook.

**How to use a notebook.** Click in a grey cell and press `Shift` + `Enter` to run it: the result appears under it. Run the cells from top to bottom, because each one uses what the cells above it made.

## Before you start: these are patient data

The results that appear under the cells are patient data, even if the patients have no name. Many patients of the autism cohort are children.

- **Before you share this notebook or send it to GitHub, clear the results:** menu *Edit*, then *Clear Outputs of All Cells*, then save.
- Never copy patient rows into an e-mail, a chat or a slide. Share counts, means and figures of groups.
- Save your own tables in the folder `data/exports/`. Everything in `data/` stays on this computer.

## 1. The catalogue of the common instruments

The first cell brings in the tools. `common` is the part of the package made for work across cohorts.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

from facedata import asp, bp, common, dr, sz

pd.set_option("display.max_columns", 30)
pd.set_option("display.max_colwidth", 90)

`common.catalogue()` gives one row per shared instrument. It was written after the schemas of the cohorts were compared variable by variable, and their values compared on the data.

- `bp`, `sz`, `dr`, `asp`: the name of the instrument in each cohort that has it (empty when the cohort does not have it);
- `stack`: **`yes`** (the same form, the same coding and the same scoring: put the cohorts under one another), **`with care`** (read the note first), **`no`** (the same name or the same subject, but not the same thing);
- `use`: the variables to analyse across cohorts;
- `note`: what differs between the cohorts.

In [ ]:
catalogue = common.catalogue()

print(len(catalogue), "instruments are shared by two cohorts or more")
catalogue[["instrument", "title", "bp", "sz", "dr", "asp", "stack"]].head(12)

How many instruments of each kind, in each chapter of the catalogue:

In [ ]:
pd.crosstab(catalogue["domain"], catalogue["stack"])

The instruments that the **four** cohorts have. `table[table["column"] == value]` keeps the rows where the column has that value.

In [ ]:
in_four = catalogue[catalogue["n_cohorts"] == 4]

print(len(in_four), "instruments are in the four cohorts")
in_four[["instrument", "title", "stack", "use"]]

**Read the note of an instrument before you use it.** The function below prints what the catalogue says. Try `"psqi"`, `"stai_ya"`, `"medications"`, `"wais4_indices"`.

In [ ]:
def about(instrument):
    """Print what the catalogue says of an instrument."""
    row = catalogue.set_index("instrument").loc[instrument]
    print(row["title"])
    print("Cohorts:", ", ".join(cohort for cohort in common.COHORTS if isinstance(row[cohort], str)))
    print("Across cohorts:", row["stack"])
    print("Variables to use:", row["use"])
    print()
    print(row["note"])


about("ymrs")

## 2. Load the common instruments of the four cohorts

We load every instrument that can be stacked (`yes` and `with care`). Two things matter:

- `common.instruments_of("bp", wanted)` gives the names to load in a cohort: an instrument does not always have the same name everywhere (the errands test is in `executive_functions` in the autism cohort);
- **`winsorise=False`**. By default the package clips the tails of the continuous variables, at limits computed inside each cohort. The same value could then be clipped in one cohort and not in another. Across cohorts, keep the values as they are.

The cell takes **about two minutes**: wait for the number in brackets on the left. To go faster, keep fewer instruments, for example `wanted = catalogue[catalogue["instrument"].isin(["identite", "cgi_egf", "ctq"])]`.

If your extraction files have other names, change them here.

In [ ]:
FILES = {"bp": "data/bp/BPALL_20261113_2.tsv", "sz": "data/sz/SZALL_20261123.tsv",
         "dr": "data/dr/DRALL_20261030.tsv", "asp": "data/asp/ASPALL_20261030.tsv"}
MODULES = {"bp": bp, "sz": sz, "dr": dr, "asp": asp}

wanted = catalogue[catalogue["stack"] != "no"]

results = {}
for cohort in common.COHORTS:
    names = common.instruments_of(cohort, wanted)
    results[cohort] = MODULES[cohort].load(FILES[cohort], instruments=names, winsorise=False)
    print(cohort, ":", len(names), "instruments,", results[cohort].data["usubjid"].nunique(), "patients,", results[cohort].data.shape[1], "columns")

Each `result` holds the same six tables as in the tutorial of its cohort: `data` (one row per visit), `patients` (one row per patient), `treatments`, `relatives`, `qc` (the catalogue of the variables) and `flags`.

## 3. The screening visit and the inclusion visit in one row

In every cohort a patient has a **screening** visit, then an **inclusion** visit (`V0`), then the follow-up visits. They are separate rows of `data`.

In [ ]:
for cohort, result in results.items():
    counts = result.data["visit"].value_counts()
    print(cohort, ":", counts["V0"], "inclusion visits,", counts.get("screening", 0), "screening visits,", counts.sum() - counts["V0"] - counts.get("screening", 0), "follow-up visits")

**An instrument may be filled at the one or at the other.** In the bipolar cohort the Mood Disorder Questionnaire is filled at the screening visit only: if you keep the inclusion rows, you lose it.

In [ ]:
bipolar = results["bp"].data

bipolar.groupby("visit")[["mdq_total", "madrs_total", "ctq_total"]].count().loc[["screening", "V0"]]

`common.baseline(result)` gives **one row per patient**. Each instrument is taken from the inclusion visit; when the inclusion visit holds nothing of an instrument, the instrument is taken whole from the screening visit. The items and the scores of an instrument therefore always come from the same visit.

It returns two tables:

- the visit, named `baseline`, with two more columns: `screening_date` and `days_from_screening`;
- the source: for each patient and each instrument, the visit its values come from (`V0`, `screening`, or empty when the instrument is filled at neither).

In [ ]:
first_bp, source_bp = common.baseline(results["bp"])

print(len(first_bp), "rows for", first_bp["usubjid"].nunique(), "patients")
print()
print("Where the MDQ comes from:")
print(source_bp["mdq"].value_counts(dropna=False))
print()
print("Where the MADRS comes from:")
print(source_bp["madrs"].value_counts(dropna=False))

We do it for the four cohorts.

In [ ]:
baselines, sources = {}, {}
for cohort, result in results.items():
    baselines[cohort], sources[cohort] = common.baseline(result)
    print(cohort, ":", len(baselines[cohort]), "patients")

**The two visits are not always close.** `days_from_screening` is the number of days from the screening visit to the inclusion visit. Look at it before you treat the baseline row as one moment in time: a mood scale filled months before the inclusion does not describe the state at inclusion.

In [ ]:
days = pd.concat(baselines.values())[["cohort", "days_from_screening"]].dropna()

summary = days.groupby("cohort")["days_from_screening"].describe()[["count", "25%", "50%", "75%"]].round(0)
summary["over 90 days, %"] = (days["days_from_screening"] > 90).groupby(days["cohort"]).mean().mul(100).round(0)
summary

If you want the screening visit only when it is close, say so: `common.baseline(result, max_days=90)` uses it only when it took place at most 90 days before or after the inclusion.

In [ ]:
close_bp, close_source = common.baseline(results["bp"], max_days=90)

print("MDQ at baseline, any screening visit:", first_bp["mdq_total"].notna().sum(), "patients")
print("MDQ at baseline, screening at most 90 days away:", close_bp["mdq_total"].notna().sum(), "patients")

## 4. Stack the four cohorts

The cohorts give the same name to the same thing, so the tables are put under one another with `pd.concat`. A column that a cohort does not have is empty for its patients.

In [ ]:
together = pd.concat(baselines.values(), ignore_index=True)
patients = pd.concat([result.patients for result in results.values()], ignore_index=True)

print(together.shape[0], "patients and", together.shape[1], "columns at baseline")
print(together.groupby("cohort")["usubjid"].nunique())

**A `usubjid` is unique inside a cohort only**: the same number can be another patient in another cohort. The key is `cohort` with `usubjid`. `fondacode`, in `patients`, is the patient code across the cohorts.

In [ ]:
print("Same usubjid in several cohorts:", patients["usubjid"].duplicated().sum())
print("Same cohort and usubjid twice:", patients.duplicated(["cohort", "usubjid"]).sum())
print("Same fondacode twice:", patients["fondacode"].duplicated().sum())

What does not change (sex, date of birth, diagnosis, the birth of the patient, the family history summed up per patient) is in `patients`. `merge` brings it next to the baseline row; the join uses the two key columns.

In [ ]:
together = together.merge(patients, on=["cohort", "usubjid"], how="left")

print(together.shape)
pd.crosstab(together["cohort"], together["diagnosis"])

## 5. Count before you compare

A column is empty for a cohort that does not have the instrument, for the children when the questionnaire is for adults, and for the years when the form was not filled. **Always count per cohort first.**

In [ ]:
scores = ["cgi_severity", "gaf", "madrs_total", "ymrs_total", "ctq_total", "psqi_total", "mars_total", "eq5d_vas", "fagerstrom_total", "staya_total", "epworth_total", "bmi"]

together.groupby("cohort")[scores].count().T

The same count for every instrument of the catalogue: the patients who hold at least one of its variables to use.

In [ ]:
rows = {}
for _, entry in wanted.iterrows():
    use = [name for name in entry["use"].split(", ") if name in together.columns]
    if use:
        filled = together[use].notna().any(axis=1)
        rows[entry["instrument"]] = filled.groupby(together["cohort"]).sum()

coverage = pd.DataFrame(rows).T.reindex(columns=common.COHORTS)
coverage.head(40)

**Look at the years.** A scale may be empty because it was not asked at that time. The childhood trauma questionnaire is in the four cohorts, but the autism cohort has it from 2020 only, and for adults.

In [ ]:
year = together["visit_date"].dt.year
has_ctq = together["ctq_total"].notna()

share = (has_ctq.groupby([together["cohort"], year]).mean() * 100).unstack("cohort")
share.plot(figsize=(9, 3), marker="o", title="Patients with the CTQ at baseline, by year of inclusion (%)")
plt.show()

## 6. A few comparisons

**The clinician's global ratings.** The CGI severity goes from 1 (normal) to 7; the GAF from 1 to 100 (higher is better). Both are in the four cohorts.

In [ ]:
print(together.groupby("cohort")[["cgi_severity", "gaf"]].agg(["count", "median"]))

together.boxplot(column="gaf", by="cohort", figsize=(7, 3))
plt.suptitle("")
plt.title("GAF at baseline, by cohort")
plt.show()

**Childhood trauma.** The 28 items are delivered on one coding and the scores are computed the same way in the four cohorts. `density=True` makes the histograms comparable although the cohorts do not have the same size.

In [ ]:
for cohort, group in together.groupby("cohort"):
    group["ctq_total"].plot.hist(bins=range(25, 126, 5), alpha=0.5, density=True, label=cohort, figsize=(8, 3))

plt.title("CTQ total at baseline")
plt.legend()
plt.show()

together.groupby("cohort")[["ctq_emotional_abuse", "ctq_physical_abuse", "ctq_sexual_abuse", "ctq_emotional_neglect", "ctq_physical_neglect"]].mean().round(1)

**Mania, with a correction to know.** The YMRS is in three cohorts. Print its note: the schizophrenia form stores three items another way, and the package puts them back on the published scale.

In [ ]:
about("ymrs")
print()
together.groupby("cohort")["ymrs_total"].describe()[["count", "mean", "50%", "75%", "max"]].round(1)

**Sleep, adherence, quality of life, body mass index.** Medians by cohort, with the number of patients behind each.

In [ ]:
several = ["psqi_total", "mars_total", "eq5d_vas", "epworth_total", "bmi", "cholesterol", "glucose"]

together.groupby("cohort")[several].agg(["count", "median"]).round(1).T

**The autism cohort has children.** For an instrument that only adults fill, or for a measure that changes with age (weight, lipids), compare adults with adults.

In [ ]:
adults = together[together["age_at_inclusion"] >= 18]

print(adults.groupby("cohort")["age_at_inclusion"].describe()[["count", "25%", "50%", "75%"]].round(0))
print()
adults.groupby("cohort")[["bmi", "cholesterol"]].median().round(1)

**Three things to keep in mind when you read these numbers.**

1. **The cohorts were not recruited the same way.** Bipolar patients are mostly seen between episodes, patients of the depression cohort in a depressive episode; the autism cohort has a group that was assessed and not diagnosed. A difference between cohorts is a description, not an effect of the diagnosis.
2. **Centres, years, age and sex are not distributed alike.** Keep `cohort`, `siteid`, the year and the age in your model.
3. **A verdict `with care` is not a detail.** The note says which part of the instrument compares.

## 7. The follow-up visits

`baseline()` only makes the first row. The follow-up visits are the rows of `data` that are neither the screening nor the inclusion. Put the baseline rows above them: `visit_month` is 0 at baseline, then 6, 12, 24...

In [ ]:
later = pd.concat([result.data[~result.data["visit"].isin(["screening", "V0"])] for result in results.values()], ignore_index=True)
every_visit = pd.concat([pd.concat(baselines.values()), later], ignore_index=True)

print(len(every_visit), "rows:", len(together), "baseline rows and", len(later), "follow-up visits")
every_visit.groupby(["visit_month", "cohort"])["gaf"].count().unstack("cohort").head(8)

In [ ]:
yearly = every_visit[every_visit["visit_month"].isin([0, 12, 24, 36])]
seen = yearly.groupby(["visit_month", "cohort"])["gaf"].agg(["count", "mean"])
seen = seen[seen["count"] >= 30]                      # a mean of a few patients says little

seen["mean"].unstack("cohort").plot(figsize=(8, 3), marker="o", title="Mean GAF at baseline and at the yearly visits")
plt.xlabel("Months since inclusion")
plt.show()

The patients seen at two years are not all the patients included: a curve of means mixes a change over time with who came back. Follow the same patients when you can.

## 8. Treatments

`treatments` has one row per treatment, with the same columns for the name, the molecule and the group in the four cohorts. **The lists are not the same thing in each cohort**: read the note before you count.

In [ ]:
about("medications")

treatments = pd.concat([result.treatments for result in results.values()], ignore_index=True)
print()
print(treatments.groupby("cohort")["usubjid"].nunique())

In `data`, sixteen columns `on_...` say whether the patient takes a kind of treatment on the day of the visit. They are computed from the list, in the bipolar, schizophrenia and depression cohorts.

In [ ]:
on = ["on_lithium", "on_anticonvulsant", "on_antipsychotic_2g", "on_antidepressant", "on_benzodiazepine"]

print(together.groupby("cohort")[on].count())
(together.groupby("cohort")[on].mean() * 100).round(0)

## 9. How much can a value be trusted?

Each cohort has a table `flags`: every value that was repaired, doubted or removed, with the reason. Stack them like the rest. `severity` has three levels: `info` (repaired, can be used), `discrepancy` (kept, but something disagrees) and `danger` (not usable: missing in `data`).

A flag names the real visit (`screening` or `V0`), not `baseline`.

In [ ]:
flags = pd.concat([result.flags for result in results.values()], ignore_index=True)

on_scores = flags[flags["variable"].isin(scores)]
on_scores.groupby(["variable", "cohort"])["severity"].value_counts().unstack("severity").fillna(0).astype(int).head(20)

## 10. What not to stack

Three instruments have the same name or the same subject in two cohorts and are not the same thing.

In [ ]:
for instrument in catalogue.loc[catalogue["stack"] == "no", "instrument"]:
    about(instrument)
    print("-" * 80)

The cohorts also measure the same subject with **other instruments**: depression with the MADRS in two cohorts, the Calgary scale in schizophrenia and the Hamilton scale or the BDI-II in autism. These scores are not on one scale: never put them in one column. The document lists them.

## 11. Save your table

Choose the columns you need and save the table in `data/exports/`. Keep `cohort` and `usubjid` (the key), and `days_from_screening` if you use an instrument that may come from the screening visit.

In [ ]:
columns = ["cohort", "usubjid", "fondacode", "sex", "age_at_inclusion", "diagnosis", "siteid", "visit_date", "days_from_screening"] + scores

my_table = together[columns]
my_table.to_csv("data/exports/common_instruments_baseline.csv", index=False)

print(len(my_table), "patients of", my_table["cohort"].nunique(), "cohorts and", my_table.shape[1], "columns saved")

## Where to go next

- `reports/executive/FACE-common-instruments.html`: every shared instrument, with the number of patients of each cohort, the years it was filled and the spread of each variable.
- `facedata/common_instruments.yaml`: the catalogue itself, and `facedata/<cohort>/differences.yaml`: what differs in a cohort, instrument by instrument.
- The notes of a schema, for the details of a cohort: `facedata/<cohort>/schemas/<tab>/<instrument>.yaml`.
- `bipolar-tutorial.ipynb`, `schizophrenia-tutorial.ipynb`, `depression-tutorial.ipynb` and `autism-tutorial.ipynb`, the tour of each cohort.

**Before you close: clear the results** (menu *Edit*, *Clear Outputs of All Cells*) if this notebook will be shared.